# 第一部分 载入库

In [1]:
import numpy as np
import pandas as pd

import cv2

import PIL
from PIL import Image
from PIL import ImageStat

from tqdm.notebook import tqdm

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

import csv

import os

import random

import collections

import shutil

from RandAugment import RandAugment

import glob

# 第二部分 数据集划分


## 2.1 BUSI

### [**整理数据集**]

1. 从原数据中分理出用于测试集、验证集

1.1 划分测试集

In [5]:
import random
import math
import tqdm

先对各类文件夹中的数量进行汇总

In [11]:
benign_adr = '..\\Data\\Dataset_BUSI_with_GT\\benign\\'
malignant_adr = '..\\Data\\Dataset_BUSI_with_GT\\malignant\\'
normal_adr = '..\\Data\\Dataset_BUSI_with_GT\\normal\\'
benign_instance_count = 0 
malignant_instance_count = 0
benign_count = 0 
malignant_count = 0
normal_count = 0

In [12]:
for file in os.listdir(os.path.join(benign_adr)):
    if 'mask' in file:
        benign_instance_count = benign_instance_count+1
    if 'mask' not in file:
        benign_count = benign_count+1
print('benign instance:',benign_instance_count)
print('benign picture:',benign_count)

benign instance: 454
benign picture: 437


In [13]:
for file in os.listdir(os.path.join(malignant_adr)):
    if 'mask' in file:
        malignant_instance_count = malignant_instance_count+1
    if 'mask' not in file:
        malignant_count = malignant_count+1
print('malignant instance:',malignant_instance_count)
print('malignant picture:',malignant_count)

malignant instance: 211
malignant picture: 210


In [14]:
for file in os.listdir(os.path.join(normal_adr)):
    if 'mask' not in file:
        normal_count = normal_count+1
print('malignant picture:',normal_count)

malignant picture: 133


Duplicated instances appear in the same picture!

由于文件命名的关系，此处根据图片进行数据集分割。 7:3的划分比例。

In [49]:
benign_train = []
malignant_train = []
normal_train = []
benign_test = []
malignant_test = []
normal_test = []

In [50]:
index = 0
while index < benign_count*0.3:
    selected = random.randrange(1,benign_count+1)
    if selected not in benign_test:
        benign_test.append(selected)
        index = index+1
index = 0
while index < malignant_count*0.3:
    selected = random.randrange(1,malignant_count+1)
    if selected not in malignant_test:
        malignant_test.append(selected)
        index = index+1
index = 0
while index < normal_count*0.3:
    selected = random.randrange(1,normal_count+1)
    if selected not in normal_test:
        normal_test.append(selected)
        index = index+1

In [51]:
len(benign_test)

132

In [52]:
len(malignant_test)

63

In [53]:
len(normal_test)

40

In [89]:
benign_test

[102,
 301,
 251,
 190,
 320,
 379,
 232,
 227,
 31,
 72,
 161,
 86,
 283,
 378,
 98,
 256,
 53,
 127,
 88,
 265,
 160,
 93,
 51,
 266,
 272,
 273,
 162,
 358,
 62,
 130,
 334,
 314,
 257,
 146,
 407,
 47,
 295,
 303,
 329,
 416,
 125,
 183,
 372,
 321,
 268,
 34,
 165,
 223,
 317,
 5,
 247,
 277,
 148,
 66,
 294,
 310,
 67,
 116,
 316,
 296,
 335,
 370,
 290,
 134,
 188,
 388,
 149,
 112,
 132,
 40,
 99,
 91,
 215,
 220,
 122,
 185,
 108,
 354,
 3,
 121,
 97,
 138,
 349,
 248,
 61,
 199,
 297,
 252,
 367,
 115,
 306,
 377,
 81,
 282,
 376,
 197,
 187,
 364,
 287,
 427,
 119,
 347,
 262,
 274,
 216,
 43,
 356,
 172,
 111,
 305,
 330,
 344,
 16,
 401,
 143,
 202,
 213,
 48,
 191,
 59,
 426,
 245,
 399,
 15,
 324,
 300,
 359,
 140,
 337,
 383,
 352,
 250]

In [65]:
#@save
def copyfile(filename, target_dir):
    """将文件复制到目标目录"""
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy(filename, target_dir)

In [60]:
save_dir = '..\\Data\\BUSI\\'
data_dir = image_address
original_dir = [data_dir+'benign\\',data_dir+'malignant\\',data_dir+'normal\\']
test_set = [benign_test,malignant_test,normal_test]
label = ['benign','malignant','normal']

In [90]:
#@save
def reorg(data_dir, save_dir): 
    for i in range(0, 3):
        for file in os.listdir(os.path.join(original_dir[i])):
            indexleft = file.find('(')
            indexright = file.find(')')
            index = int(file[indexleft+1:indexright])
            if index in test_set[i]:
                if 'mask' in file:
                    copyfile(os.path.join(data_dir,label[i],file), os.path.join(save_dir,'test','mask',label[i]))
                else:
                    copyfile(os.path.join(data_dir,label[i],file), os.path.join(save_dir,'test','image',label[i]))
            else:
                if 'mask' in file:
                    copyfile(os.path.join(data_dir,label[i],file), os.path.join(save_dir,'train','mask',label[i]))
                else:
                    copyfile(os.path.join(data_dir,label[i],file), os.path.join(save_dir,'train','image',label[i]))
        print(label[i]+' finished')

In [91]:
reorg(data_dir, save_dir)

benign finished
malignant finished
normal finished


## 2.2 BUSBRA

In [8]:
busbra_adr = '../../Data/BreastUltrasound/BUSBRA/Images'

In [9]:
files = os.listdir(busbra_adr)
# 过滤出文件（排除文件夹）
busbra_num = len(files)
busbra_num

1875

In [14]:
selected_index = []
while len(selected_index)<busbra_num*0.4:
    random_number = random.randint(0, busbra_num-1)
    if random_number not in selected_index:
        selected_index.append(random_number)
print(len(selected_index))

750


In [15]:
import shutil
#@save
def copyfile(filename, target_dir):
    """将文件复制到目标目录"""
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy(filename, target_dir)

In [16]:
busbra_adr.replace('Images','Masks')

'../../Data/BreastUltrasound/BUSBRA/Masks'

In [20]:
#Valid Division
images = glob.glob(busbra_adr + "/*")
masks = glob.glob(busbra_adr.replace('Images','Masks') + "/*")
for j in range(len(selected_index)):
    old_image_path = images[selected_index[j]]
    new_image_path = os.path.join(busbra_adr.replace('Images','valid/image'))
    copyfile(old_image_path,new_image_path)
    old_mask_path = masks[selected_index[j]]
    new_mask_path = os.path.join(busbra_adr.replace('Images','valid/mask'))
    copyfile(old_mask_path,new_mask_path)

In [22]:
#Train Division
images = glob.glob(busbra_adr + "/*")
masks = glob.glob(busbra_adr.replace('Images','Masks') + "/*")
for j in range(len(images)):
    if j not in selected_index:
        old_image_path = images[j]
        new_image_path = os.path.join(busbra_adr.replace('Images','train/image'))
        copyfile(old_image_path,new_image_path)
        old_mask_path = masks[j]
        new_mask_path = os.path.join(busbra_adr.replace('Images','train/mask'))
        copyfile(old_mask_path,new_mask_path)

## 2.3 USG

In [123]:
usg_adr = '../../Data/BreastUltrasound/USG/'

In [125]:
df = pd.read_excel(usg_adr+'BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx')
# 根据行和列范围提取数据
selected_data = df.loc[:, :]
print(selected_data)

     CaseID Image_filename Mask_tumor_filename Mask_other_filename  \
0         1    case001.png   case001_tumor.png                 NaN   
1         2    case002.png   case002_tumor.png                 NaN   
2         3    case003.png   case003_tumor.png                 NaN   
3         4    case004.png   case004_tumor.png                 NaN   
4         5    case005.png   case005_tumor.png                 NaN   
..      ...            ...                 ...                 ...   
251     252    case252.png   case252_tumor.png                 NaN   
252     253    case253.png   case253_tumor.png                 NaN   
253     254    case254.png   case254_tumor.png                 NaN   
254     255    case255.png   case255_tumor.png                 NaN   
255     256    case256.png   case256_tumor.png                 NaN   

     Pixel_size            Age                           Tissue_composition  \
0      0.007812             57             heterogeneous: predominantly fat   
1

In [126]:
usg_mask_adr =  '../../Data/BreastUltrasound/USG/mask'
usg_img_adr = '../../Data/BreastUltrasound/USG/image'

In [137]:
#@save
def copyfile(filename, target_dir):
    """将文件复制到目标目录"""
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy(filename, target_dir)

对于无异常图和多异常图进行处理 255《522/2

In [129]:
for i in range(selected_data.shape[0]):
    if str(selected_data.iloc[i,2]) == 'nan':
        # 全黑Mask生成
        image =  cv2.imread(os.path.join(usg_adr,'files',selected_data.iloc[i,1]))
        mask = np.zeros_like(image)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        dot_index = selected_data.iloc[i,1].find('.')
        mask_name = selected_data.iloc[i,1][:dot_index]+'_tumor'+selected_data.iloc[i,1][dot_index:]
        print(mask_name,'- 0')
        cv2.imwrite(os.path.join(usg_mask_adr,mask_name), mask)
        image_adr = os.path.join(usg_adr,'files',selected_data.iloc[i,1])
        copyfile(image_adr,usg_img_adr)
    else:
        if str(selected_data.iloc[i,3]) != 'nan':
            # 合并Mask图
            other_files = selected_data.iloc[i,3]
            other_files_nums = int(other_files.count('&'))+1
            other_files_target = []
            if other_files_nums == 1:
                other_files_target.append(other_files)
            else:
                while other_files.find('&')!= -1:
                    position_index = other_files.find('&')
                    other_file = other_files[0:position_index]
                    other_files_target.append(other_file)
                    other_files = other_files[position_index+1:]
                other_files_target.append(other_files)
            other_files_target.append(selected_data.iloc[i,2])
            for j in range(len(other_files_target)):
                if j == 0:
                    mask = cv2.imread(os.path.join(usg_adr,'files',other_files_target[j]),0)
                else:
                    mask_other = cv2.imread(os.path.join(usg_adr,'files',other_files_target[j]),0)
                    mask = cv2.bitwise_or(mask, mask_other)
            mask_name = selected_data.iloc[i,2]
            print(mask_name,'-',other_files_nums)
            cv2.imwrite(os.path.join(usg_mask_adr,mask_name), mask)
            image_adr = os.path.join(usg_adr,'files',selected_data.iloc[i,1])
            copyfile(image_adr,usg_img_adr)
        else:
            image =  cv2.imread(os.path.join(usg_adr,'files',selected_data.iloc[i,1]))
            blank_mask = np.zeros_like(image)
            blank_mask = cv2.cvtColor(blank_mask, cv2.COLOR_BGR2GRAY)
            mask = cv2.imread(os.path.join(usg_adr,'files',selected_data.iloc[i,2]),0)
            mask = cv2.bitwise_or(mask, blank_mask)
            mask_name = selected_data.iloc[i,2]
            cv2.imwrite(os.path.join(usg_mask_adr,mask_name), mask)
#             print(mask.shape)
            image_adr = os.path.join(usg_adr,'files',selected_data.iloc[i,1])
#             mask_adr = os.path.join(usg_adr,'files',selected_data.iloc[i,2])
            copyfile(image_adr,usg_img_adr)
#             copyfile(mask_adr,usg_mask_adr)

case022_tumor.png - 2
case036_tumor.png - 2
case038_tumor.png - 1
case045_tumor.png - 0
case061_tumor.png - 0
case085_tumor.png - 2
case092_tumor.png - 1
case140_tumor.png - 4
case151_tumor.png - 2
case209_tumor.png - 0
case213_tumor.png - 0


In [130]:
files = os.listdir(usg_img_adr)
# 过滤出文件（排除文件夹）
usg_num = len(files)
usg_num

256

In [134]:
selected_index = []
while len(selected_index)<usg_num*0.4:
    random_number = random.randint(0, usg_num-1)
    if random_number not in selected_index:
        selected_index.append(random_number)
print(len(selected_index))

103


In [138]:
#Valid Division
images = glob.glob(usg_img_adr + "/*")
masks = glob.glob(usg_mask_adr + "/*")
for j in range(len(selected_index)):
    old_image_path = images[selected_index[j]]
    new_image_path = os.path.join(usg_img_adr.replace('image','valid/image'))
    copyfile(old_image_path,new_image_path)
    old_mask_path = masks[selected_index[j]]
    new_mask_path = os.path.join(usg_mask_adr.replace('mask','valid/mask'))
    copyfile(old_mask_path,new_mask_path)

## 2.4  BUS Dataset B

In [2]:
busbset_adr = '../../Data/BreastUltrasound/BUSBset/'

In [5]:
files = os.listdir(busbset_adr+'original')
# 过滤出文件（排除文件夹）
busbset_num = len(files)
busbset_num

163

In [6]:
selected_index = []
while len(selected_index)<busbset_num*0.4:
    random_number = random.randint(0, busbset_num-1)
    if random_number not in selected_index:
        selected_index.append(random_number)
print(len(selected_index))

66


In [7]:
import shutil
#@save
def copyfile(filename, target_dir):
    """将文件复制到目标目录"""
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy(filename, target_dir)

In [8]:
#Valid Division
images = glob.glob(busbset_adr + "original/*")
masks = glob.glob(busbset_adr + "GT/*")
for j in range(len(selected_index)):
    old_image_path = images[selected_index[j]]
    new_image_path = busbset_adr + "test/image/"
    copyfile(old_image_path,new_image_path)
    old_mask_path = masks[selected_index[j]]
    new_mask_path = busbset_adr + "test/mask/"
    copyfile(old_mask_path,new_mask_path)

In [9]:
#Train Division
images = glob.glob(busbset_adr + "original/*")
masks = glob.glob(busbset_adr + "GT/*")
for j in range(len(images)):
    if j not in selected_index:
        old_image_path = images[j]
        new_image_path = busbset_adr + "train/image/"
        copyfile(old_image_path,new_image_path)
        old_mask_path = masks[j]
        new_mask_path = busbset_adr + "train/mask/"
        copyfile(old_mask_path,new_mask_path)